# Remez-Design (Chebyshev-Approximation)

Beim Entwurf von FIR-Filtern unter Verwendung der Chebyshev-Approximation wurde gezeigt, dass diese Approximation einen großen Vorteil bietet, da sie das gesamte Toleranzschema ausnutzt. Eine solche Entwurfsmethode (Anwendung der Chebyshev-Approximation) für FIR-Systeme ist in Python in der Funktion `scipy.signal.remez()` (Parks–McClellan-Algorithmus) implementiert. Die Approximation erfolgt in den Pass- und Stop-Bändern.

Der Aufruf von `b = remez(m, F, G)` entwirft einen Filter der Ordnung \(m\), dessen Spezifikation durch die Vektoren `F` und `G` vorgegeben ist. `F` enthält Paare von Frequenzen (zwischen 0 und 1), die die Pass- oder Stop-Bänder festlegen, und `G` ist die Frequenzantwort (Amplitude), die den Frequenzen in `F` entspricht.

**Beispiel:**
```python
from scipy.signal import remez

F = [0, 0.3, 0.5, 1]
G = [1, 1, 0, 0]

b = remez(10, F, G)  # Beispiel: Filter 10. Ordnung

## Aufgaben

**a)**  
Berechnen Sie einen Filter mit der gegebenen Frequenzantwort. Experimentieren Sie mit verschiedenen Filterordnungen und zählen Sie die Anzahl der Extremwerte innerhalb des Approximationsintervalls.

**b)**  
Der Aufruf von `b = remez(m, F, G, W)` gewichtet die gewünschte Frequenzantwort, wobei Frequenzbänder mit niedrigerem Gewicht weniger genau approximiert werden als Frequenzbänder mit höherem Gewicht.  
Die Anzahl der Gewichte im Vektor `W` entspricht der Hälfte der Anzahl der Einträge in `F` oder `G` (für jedes Frequenzband ein Gewichtsfaktor).

Das Frequenzband zwischen 0,0 und 0,3 (Passband) muss die gewünschte Frequenzantwort nicht sehr genau approximieren. Für das Frequenzband zwischen 0,5 und 1 sollte ein höheres Gewicht verwendet werden.  
Berechnen Sie den Filter erneut und vergleichen Sie die Frequenzantwort mit dem Filter aus Aufgabe a).

**c)**  
Entwerfen Sie die Low-Pass – High-Pass-Kombination wie unten angegeben, wobei die Passbänder (um den Faktor 4) höher gewichtet werden sollen als die Stopbänder:

| Band        | Frequenzbereich |
|-------------|----------------|
| Passband 1  | 0 – 0.2        |
| Stopband 1  | 0,26 – 0,44    |
| Passband 2  | 0,5 – 0,7      |
| Stopband 2  | 0,76 – 1       |

Filterordnung `m`: 128

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import remez, freqz
# BEGIN SOLUTION
# #####  Part a: Design Filters  ######################################
f = [0, 0.3, 0.5, 1.0]  # Frequencies of passband and stopband
g = [1, 0]              # Desired frequency response
w = [1,1]
v_m = [9, 10]           # Vector with filter orders

f_plot = [f[0], f[1], f[2], f[3]]
g_plot = [g[0], g[0], g[1], g[1]]

for m in v_m:
    b = remez(m + 1, f, g, weight=w, fs=2.0)
    W, H = freqz(b, worN=512)  # Frequency response calculation

    # # Graphical output # #
    string = f'Desired/Actual Response - Filter Order: {m}'
    plt.figure()
    plt.plot(f_plot, g_plot, label='Desired Response')
    plt.plot(W / np.pi, np.abs(H), label='Actual Response')
    plt.axhline(y=2 - np.max(np.abs(H)), xmin=0, xmax=0.3, linestyle='--')
    plt.axhline(y=np.max(np.abs(H)), xmin=0, xmax=0.5, linestyle='--')
    plt.axhline(y=np.max(np.abs(H[400:])), xmin=0.5, xmax=1, linestyle='--')
    plt.axvline(x=0.3, ymin=2 - np.max(np.abs(H)), ymax=0, linestyle='--')
    plt.axvline(x=0.5, ymin=np.max(np.abs(H)), ymax=np.max(np.abs(H[400:])), linestyle='--')
    plt.grid()
    plt.xlabel('Omega/pi')
    plt.ylabel('|H(exp(j*Omega))|')
    plt.title(string)
    plt.legend()

# #####  Part b: Weight the Frequency Response  ##################################
f = [0, 0.3, 0.5, 1.0]  # Frequencies of passband and stopband
g = [1, 0]              # Desired frequency response
w = [0.5, 2]            # Weighting of passband and stopband
v_m = [9, 10]           # Vector with filter orders

f_plot = [f[0], f[1], f[2], f[3]]
g_plot = [g[0], g[0], g[1], g[1]]

for m in v_m:
    b = remez(m + 1, f, g, weight=w, fs=2.0)
    W, H = freqz(b, worN=512)
    
    # # Graphical output # #
    string = f'Desired/Actual Response - Filter Order: {m}'
    plt.figure()
    plt.plot(f_plot, g_plot, label='Desired Response')
    plt.plot(W / np.pi, np.abs(H), label='Actual Response')
    plt.axhline(y=2 - np.max(np.abs(H)), xmin=0, xmax=0.3, linestyle='--')
    plt.axhline(y=np.max(np.abs(H)), xmin=0, xmax=0.5, linestyle='--')
    plt.axhline(y=np.max(np.abs(H[400:])), xmin=0.5, xmax=1, linestyle='--')
    plt.axvline(x=0.3, ymin=2 - np.max(np.abs(H)), ymax=0, linestyle='--')
    plt.axvline(x=0.5, ymin=np.max(np.abs(H)), ymax=np.max(np.abs(H[400:])), linestyle='--')
    plt.grid()
    plt.xlabel('Omega/pi')
    plt.ylabel('|H(exp(j*Omega))|')
    plt.title(string)
    plt.legend()

# #####  Part c: Low-pass Band-pass Combination  #########################
f = [0, 0.2, 0.26, 0.44, 0.5, 0.7, 0.76, 1]  # Vector of corner frequencies
g = [1,0,1,0]                 # Desired frequency response
w = [1, 4, 1, 4]              # Vector with weights
m = 128                       # Filter order

b = remez(m + 1, f, g, weight=w, fs=2.0)
W, H = freqz(b, worN=512)

# # Graphical output # #
plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H)))
plt.grid()
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))| in dB')
plt.title('LP-BP Combination')

plt.show()
# END SOLUTION

# Lösung 5.3: Remez-Design (Chebyshev-Approximation)

## Teil a
Beide dargestellten Amplitudenantworten sind Beispiele für den Entwurf linearer Tiefpassfilter unter Verwendung der Chebyshev-Approximation im Passband und Stopband.  
Der Entwurf erfolgt mithilfe des **Remez-Designs**, wobei die Intervalle `[0, 0.3]` und `[0.5, 1]` festgelegt sind.  
Die Filterordnungen sind `m = 9` und `m = 10`.

<img src="./pngs/ldsv5a03_a1.png" width="300px" style="display:inline-block; margin-right:10px;">
<img src="./pngs/ldsv5a03_a2.png" width="300px" style="display:inline-block;">

Beide Plots bestätigen die gleichmäßige Approximation im Passband und Stopband.  
Die Anzahl der Extremwerte, die das Toleranzschema erfüllen, einschließlich der Randwerte, ist `floor(m/2) + 2` für `m = 9, 10` in beiden Amplitudenantworten.

## Teil b
Wenn das Stopband stärker gewichtet wird als das Passband (z. B. Faktor 4), ergeben sich für die gleichen Randbedingungen wie in Teil a folgende Frequenzantworten:

<img src="./generated_notebooks/pngs/ldsv5a03_b1.png" width="300px" style="display:inline-block; margin-right:10px;">
<img src="./generated_notebooks/pngs/ldsv5a03_b2.png" width="300px" style="display:inline-block;">

Ein Vergleich mit Teil a verdeutlicht den Einfluss der Gewichtung auf Stopband und Passband.

## Teil c
Aufgrund der gegebenen Frequenzbänder, Frequenzantworten und Gewichtsfaktoren ist der Entwurf selektiver Systeme mit dem **Remez-/firpm-Design** problemlos möglich.  
Das folgende Beispiel zeigt den Entwurf einer Tiefpass–Bandpass-Kombination, wie in der Aufgabe spezifiziert:

<img src="./generated_notebooks/pngs/ldsv5a03_c1.png" width="300px">